> **TrustBreast — Notebook 7.** Out-of-fold evaluation of the cross-conformal coverage (WBCD and Coimbra), and a check of which constituent errors fall inside the MC-Dropout escalation set.

## Chalane ka tareeqa
1. Runtime → Change runtime type → **CPU** → Save
2. Runtime → **Run all** (warning aaye to *Run anyway*)
3. Waqt: ~20–30 minute
4. Sab se aakhri cell (**RESULTS SUMMARY**) aur us se pehle wale flag-check cell ka output bhejein.

In [ ]:
# Colab: repo clone karo (models/ folder isi mein hai). Local Jupyter par yeh cell kuch nahi karta.
import os
if os.path.exists('/content') and not os.path.isdir('models') and not os.path.isdir('../models'):
    !git clone -q https://github.com/Iqra672-ai/TrustBreast.git /content/TrustBreast
    %cd /content/TrustBreast
    !pip -q install -r requirements.txt


In [ ]:
# ============================================
# CELL 0 — DETERMINISM  (SAB SE PEHLE chalao, imports se bhi pehle)
# Yeh 3 cheezein add karta hai jo DNN ko har run SAME banati hain:
#   1. os.environ flags   -> GPU/cuDNN ko deterministic (imports se PEHLE set hona zaroori)
#   2. saare seeds        -> python / numpy / tensorflow
#   3. enable_op_determinism() -> GPU floating-point order LOCK (yehi asal missing cheez thi)
# reseed() helper baad me DNN se theek pehle RNG ko wapas fix karta hai.
# ============================================
import os
os.environ['PYTHONHASHSEED']         = '42'
os.environ['TF_DETERMINISTIC_OPS']   = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import random, numpy as np, tensorflow as tf
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("enable_op_determinism() ON  ->  DNN ab reproducible")
except Exception as e:
    print("Note: enable_op_determinism unavailable (purani TF). Baqi fixes phir bhi lagenge.")

def reseed(s=SEED):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)

print("TF:", tf.__version__, "| Determinism setup done. Ab baqi cells chalao.")


In [ ]:
import pandas as pd, numpy as np
from sklearn.preprocessing import LabelEncoder
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
col_names = ['id','diagnosis',
  'radius_mean','texture_mean','perimeter_mean','area_mean','smoothness_mean','compactness_mean',
  'concavity_mean','concave_points_mean','symmetry_mean','fractal_dimension_mean',
  'radius_se','texture_se','perimeter_se','area_se','smoothness_se','compactness_se',
  'concavity_se','concave_points_se','symmetry_se','fractal_dimension_se',
  'radius_worst','texture_worst','perimeter_worst','area_worst','smoothness_worst',
  'compactness_worst','concavity_worst','concave_points_worst','symmetry_worst','fractal_dimension_worst']
df = pd.read_csv(url, header=None, names=col_names)
df['diagnosis'] = LabelEncoder().fit_transform(df['diagnosis'])   # B=0, M=1
X = df.drop(['id','diagnosis'], axis=1); y = df['diagnosis']
feature_names = list(X.columns)
print(f"Data: {len(df)} patients | B: {(y==0).sum()} | M: {(y==1).sum()}")


In [ ]:
# ===== LOAD the locked 99.12% model (run this FIRST) =====
# Requires the saved model folder in Google Drive: MyDrive/TrustBreast_locked/
# (produced once by File 1 - Objective 1). No retraining here, so the number is always 99.12%.
import os, pickle, joblib, numpy as np, tensorflow as tf
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
# Locked model: pehle GitHub repo ka models/ folder, warna Google Drive
SAVE_DIR = next((p for p in ['models/TrustBreast_locked', '../models/TrustBreast_locked']
                 if os.path.isdir(p)), None)
if SAVE_DIR is None:
    SAVE_DIR = '/content/drive/MyDrive/TrustBreast_locked'
    from google.colab import drive; drive.mount('/content/drive')
print('Loading locked model from:', SAVE_DIR)
rf_model  = joblib.load(SAVE_DIR + '/rf_model.pkl')
xgb_model = joblib.load(SAVE_DIR + '/xgb_model.pkl')
scaler    = joblib.load(SAVE_DIR + '/scaler.pkl')
dnn_best  = tf.keras.models.load_model(SAVE_DIR + '/dnn_best.keras')
dnn_model = tf.keras.models.load_model(SAVE_DIR + '/dnn_model.keras')
with open(SAVE_DIR + '/state.pkl','rb') as f: state = pickle.load(f)
globals().update({k:v for k,v in state.items() if v is not None})
if globals().get('prob_ensemble_val') is None and 'X_val_sc' in globals():
    _rf=rf_model.predict_proba(X_val_sc)[:,1]; _xg=xgb_model.predict_proba(X_val_sc)[:,1]
    _dn=dnn_best.predict(X_val_sc, verbose=0).ravel(); prob_ensemble_val=(_rf+_xg+_dn)/3
model_dnn=dnn_best; rf_aug=rf_model; xgb_aug=xgb_model; feature_names=list(X.columns)
print('LOADED locked model. Ensemble accuracy:', round(accuracy_score(y_test, ens_pred)*100,2), 'percent')

## Which errors fall inside the 12 flagged patients?

In [ ]:
# Are the Table 2 errors of each constituent inside the 12 flagged patients?
import pandas as pd, numpy as np
o3 = pd.read_csv('results/O3_all_patients.csv').sort_values('patient_idx')
flag = set(o3.loc[o3.mc_std > 0.15, 'patient_idx']); yt = np.asarray(y_test).ravel().astype(int)
for name, prob, thr in [('RF', rf_prob_raw, best_rf_thr), ('XGBoost', xgb_prob_raw, best_xgb_thr),
                        ('DNN (Table 2, locked model)', dnn_prob_raw, best_dnn_thr), ('Ensemble', prob_ensemble, best_ens_thr)]:
    err = set(np.where((np.asarray(prob).ravel() >= thr).astype(int) != yt)[0])
    print(f"{name:<28} thr {thr:.2f}  errors {sorted(err)}  inside flagged-12: {sorted(err & flag)}  outside: {sorted(err - flag)}")


## WBCD — same 10-fold CV as notebook 01 (must give 96.49 ± 2.74)

In [ ]:
# ============================================================
#  FULL-ENSEMBLE LEAKAGE-CORRECTED 10-FOLD CROSS-VALIDATION
#  10-Fold CV of the full RF + XGBoost + DNN soft-voting ensemble
#
#  Notebook mein iss cell ko File-1 ke LOAD/DATA cell ke BAAD chalao
#  (X, y already bane hue hone chahiye — poora 569-patient dataset).
#
#  Scaler + SMOTE sirf har fold ke training hisse par fit hote hain (no leakage).
#  Har fold ka DNN keras.utils.set_random_seed(SEED + fold) se seed hota hai -> har run SAME.
#  Poora 569-patient dataset; std = sample std (ddof=1).
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.initializers import GlorotUniform
from matplotlib.patches import Patch

SEED = 42
tf.keras.utils.set_random_seed(SEED)   # FIX: Keras 3 ka global RNG bhi fix (python+numpy+tf+keras)

# Poora dataset (569 patients)
X_arr = np.array(X)
y_arr = np.array(y)

N_FOLDS = 10
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_accs = []
oof_true, oof_prob = [], []   # out-of-fold collectors
print("=" * 60)
print(f"FULL-ENSEMBLE LEAKAGE-CORRECTED {N_FOLDS}-FOLD CV")
print("=" * 60)
print(f"Total patients used in CV: {len(y_arr)}")
print()

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_arr, y_arr), 1):
    X_tr_raw, X_va_raw = X_arr[tr_idx], X_arr[va_idx]
    y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

    # ---- scaler fit ONLY on this fold's train portion ----
    fold_scaler = MinMaxScaler()
    X_tr_sc = fold_scaler.fit_transform(X_tr_raw)
    X_va_sc = fold_scaler.transform(X_va_raw)          # validation never seen by fit

    # ---- SMOTE ONLY on this fold's train portion ----
    sm = SMOTE(random_state=SEED)
    X_tr_sm, y_tr_sm = sm.fit_resample(X_tr_sc, y_tr)

    # ---- Random Forest ----
    rf = RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=1)
    rf.fit(X_tr_sm, y_tr_sm)
    rf_p = rf.predict_proba(X_va_sc)[:, 1]

    # ---- XGBoost ----
    xgb = XGBClassifier(learning_rate=0.01, max_depth=4, n_estimators=500,
                        subsample=0.9, colsample_bytree=0.9, random_state=SEED,
                        n_jobs=1, eval_metric='logloss', verbosity=0)
    xgb.fit(X_tr_sm, y_tr_sm)
    xgb_p = xgb.predict_proba(X_va_sc)[:, 1]

    # ---- DNN (fresh network each fold, seeded) ----
    tf.keras.utils.set_random_seed(SEED + fold)   # FIX: har fold ka DNN har run par SAME
    dnn = Sequential([
        Dense(256, activation='relu', kernel_regularizer=l2(0.0005),
              kernel_initializer=GlorotUniform(seed=SEED), input_shape=(X_tr_sm.shape[1],)),
        BatchNormalization(), Dropout(0.3),
        Dense(128, activation='relu', kernel_regularizer=l2(0.0005),
              kernel_initializer=GlorotUniform(seed=SEED + 1)),
        BatchNormalization(), Dropout(0.3),
        Dense(64, activation='relu', kernel_initializer=GlorotUniform(seed=SEED + 2)),
        BatchNormalization(), Dropout(0.2),
        Dense(1, activation='sigmoid', kernel_initializer=GlorotUniform(seed=SEED + 3)),
    ])
    dnn.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy')
    dnn.fit(X_tr_sm, y_tr_sm, epochs=80, batch_size=16, verbose=0,
            validation_split=0.15,
            callbacks=[EarlyStopping(patience=12, restore_best_weights=True)])
    dnn_p = dnn.predict(X_va_sc, verbose=0).ravel()

    # ---- soft-vote ensemble (simple average, matches Objective-1) ----
    ens_p = (rf_p + xgb_p + dnn_p) / 3
    ens_pred_fold = (ens_p >= 0.5).astype(int)
    acc = accuracy_score(y_va, ens_pred_fold)
    fold_accs.append(acc)
    oof_true.extend(list(np.asarray(y_va).ravel()))
    oof_prob.extend(list(np.asarray(ens_p).ravel()))
    print(f"Fold {fold}: ensemble accuracy = {acc*100:.2f}%  "
          f"(RF {accuracy_score(y_va,(rf_p>=0.5).astype(int))*100:.2f}% | "
          f"XGB {accuracy_score(y_va,(xgb_p>=0.5).astype(int))*100:.2f}% | "
          f"DNN {accuracy_score(y_va,(dnn_p>=0.5).astype(int))*100:.2f}%)")

# ============================================================
#  RESULT  (sample std = ddof=1, image jaisa)
# ============================================================
fold_accs = np.array(fold_accs)
mean_acc = fold_accs.mean() * 100
std_acc  = fold_accs.std(ddof=1) * 100          # sample std (ddof=1)

print()
print("=" * 60)
print(f"{N_FOLDS}-FOLD CV RESULT (full ensemble, leakage-free)")
print("=" * 60)
print(f"Mean accuracy: {mean_acc:.2f}%")
print(f"Std deviation: {std_acc:.2f}%")
print(f"-> Report as: {mean_acc:.2f}% +/- {std_acc:.2f}%")

# ---- pooled out-of-fold AUC + bootstrap 95% CI (asli numbers) ----
from sklearn.metrics import roc_auc_score
oof_true = np.array(oof_true); oof_prob = np.array(oof_prob)
oof_pred = (oof_prob >= 0.5).astype(int)
pooled_auc = roc_auc_score(oof_true, oof_prob)
rng = np.random.RandomState(42); N = len(oof_true); boot = []
for _ in range(2000):
    b = rng.randint(0, N, N)
    boot.append((oof_pred[b] == oof_true[b]).mean())
ci_low, ci_high = np.percentile(boot, [2.5, 97.5]) * 100
print()
print(f"Pooled out-of-fold AUC : {pooled_auc:.4f}")
print(f"Bootstrap 95% CI       : {ci_low:.2f}% - {ci_high:.2f}%")
print()
print("=> Thesis 4.2.2 me bharo:")
print(f"   {mean_acc:.2f}% +/- {std_acc:.2f}%  |  95% CI {ci_low:.2f}-{ci_high:.2f}%  |  pooled AUC {pooled_auc:.4f}")

# ============================================================
#  FIGURE 4.6  ->  bar chart (green = above mean, red = below)
# ============================================================
accs   = fold_accs * 100
folds  = [f"Fold {i+1}" for i in range(len(accs))]
colors = ['#2ca02c' if a >= mean_acc else '#e74c3c' for a in accs]

fig, ax = plt.subplots(figsize=(12, 5.5))
bars = ax.bar(folds, accs, color=colors, edgecolor='black', width=0.6, zorder=3)
ax.axhline(mean_acc, ls='--', color='#333', lw=1.5, zorder=2)

for b, a in zip(bars, accs):
    ax.text(b.get_x() + b.get_width() / 2, a + 0.12, f'{a:.2f}%',
            ha='center', va='bottom', fontweight='bold', fontsize=9)

ax.set_ylim(accs.min() - 2.0, 101.0)
ax.set_ylabel('Accuracy (%)')
ax.set_title(f"Full-Ensemble Leakage-Corrected {N_FOLDS}-Fold Cross-Validation\n"
             f"Mean = {mean_acc:.2f}% \u00b1 {std_acc:.2f}%  (RF + XGBoost + DNN soft-voting)",
             fontsize=11, fontweight='bold', pad=15)
ax.grid(axis='y', alpha=0.3, zorder=0)

ax.legend(handles=[Patch(color='#2ca02c', label='Above mean'),
                   Patch(color='#e74c3c', label='Below mean'),
                   plt.Line2D([0], [0], ls='--', color='#333',
                              label=f'Mean = {mean_acc:.2f}%')],
          loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('figure_4_6_cv_10fold.png', dpi=200, bbox_inches='tight')
plt.close()
print("\nChart saved as: figure_4_6_cv_10fold.png")

# handy variables for later cells
cv_full_ensemble_mean   = fold_accs.mean()
cv_full_ensemble_std    = fold_accs.std(ddof=1)
cv_full_ensemble_scores = fold_accs


In [ ]:
# ---------- out-of-fold (fold-held-out) conformal evaluation ----------
import math, numpy as np
def kth(sc, alpha):
    sc = np.sort(sc); n = len(sc); k = math.ceil((n + 1) * (1 - alpha)); return sc[min(k, n) - 1], k, n

def conformal_report(y, p, fold, alpha, label):
    """y: labels (1 = positive class), p: out-of-fold P(positive), fold: fold id of each patient."""
    y = np.asarray(y).astype(int); p = np.asarray(p, float); fold = np.asarray(fold)
    s = np.where(y == 1, 1 - p, p)                       # nonconformity score, Eq. (4)
    # (a) in-sample: threshold and coverage from the same 569 scores (what the paper reported)
    t, k, n = kth(s, alpha); ins = np.mean(s <= t)
    # (b) out-of-fold: for each fold, threshold from the OTHER folds only, applied to this fold
    cov = np.zeros(len(y), bool); size = np.zeros(len(y), int)
    covM = np.zeros(len(y), bool)
    for f in np.unique(fold):
        te, ca = fold == f, fold != f
        tj, _, _ = kth(s[ca], alpha)                      # marginal threshold
        tb, _, _ = kth(s[ca & (y == 0)], alpha)           # Mondrian thresholds
        tm, _, _ = kth(s[ca & (y == 1)], alpha)
        inc0, inc1 = p[te] <= tj, (1 - p[te]) <= tj
        cov[te] = np.where(y[te] == 1, inc1, inc0); size[te] = inc0.astype(int) + inc1.astype(int)
        covM[te] = np.where(y[te] == 1, (1 - p[te]) <= tm, p[te] <= tb)
    print(f"{label}  alpha={alpha:.2f}")
    print(f"   in-sample (paper so far): tau {t:.4f}  index {k}/{n}  coverage {ins*100:.2f}%  (= index/n by construction)")
    print(f"   OUT-OF-FOLD marginal coverage {cov.mean()*100:.2f}%  | positive {cov[y==1].mean()*100:.2f}%  negative {cov[y==0].mean()*100:.2f}%")
    print(f"   OUT-OF-FOLD sets single/ambiguous/empty = {(size==1).sum()}/{(size==2).sum()}/{(size==0).sum()}")
    print(f"   OUT-OF-FOLD Mondrian coverage  positive {covM[y==1].mean()*100:.2f}%  negative {covM[y==0].mean()*100:.2f}%")
    return dict(ins=ins, cov=cov.mean(), cpos=cov[y==1].mean(), cneg=cov[y==0].mean(),
                sets=((size==1).sum(), (size==2).sum(), (size==0).sum()), mpos=covM[y==1].mean(), mneg=covM[y==0].mean())


In [ ]:
# fold id of every out-of-fold prediction (oof_* were collected fold by fold, in skf order)
fold_id = np.concatenate([np.full(len(va), f) for f, (_, va) in enumerate(skf.split(X_arr, y_arr), 1)])
assert len(fold_id) == len(oof_true)
print(f"CV check (must equal the paper): {cv_full_ensemble_mean*100:.2f} +/- {cv_full_ensemble_std*100:.2f}, pooled AUC {pooled_auc:.4f}   [paper: 96.49 +/- 2.74, 0.9945]")
W = {a: conformal_report(oof_true, oof_prob, fold_id, a, "WBCD (569, malignant = positive)") for a in (0.05, 0.10)}


## Coimbra — same 5-fold CV as notebook 06 (must give 70.62 ± 14.35)

In [ ]:
import pandas as pd, numpy as np
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00451/dataR2.csv"
import os
local = next((f for f in ['data/coimbra_dataR2.csv', '../data/coimbra_dataR2.csv', 'dataR2.csv'] if os.path.exists(f)), None)
try:
    df = pd.read_csv(local if local else URL)
    print("Data source:", local if local else URL)
except Exception as e:
    print("Direct download failed, trying ucimlrepo:", str(e)[:60])
    import subprocess; subprocess.run(['pip', '-q', 'install', 'ucimlrepo'])
    from ucimlrepo import fetch_ucirepo
    r = fetch_ucirepo(id=451); df = pd.concat([r.data.features, r.data.targets], axis=1)
target = 'Classification'
X_c = df.drop(columns=[target]).astype(float).values
y_c = (df[target].values == 2).astype(int)          # 1 = breast cancer, 0 = healthy control
FEATS_C = list(df.drop(columns=[target]).columns)
print(f"Coimbra: {len(y_c)} patients | cancer {y_c.sum()} | controls {(y_c == 0).sum()} | features {len(FEATS_C)}: {FEATS_C}")
assert len(y_c) == 116 and y_c.sum() == 64


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, recall_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

N_FOLDS = 5
def build_dnn(dim, seed):
    tf.keras.utils.set_random_seed(seed)
    m = tf.keras.Sequential([tf.keras.layers.Input(shape=(dim,))])
    for u, dr, reg in [(256, .3, 5e-4), (128, .3, 5e-4), (64, .2, None)]:
        m.add(tf.keras.layers.Dense(u, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(reg) if reg else None))
        m.add(tf.keras.layers.BatchNormalization()); m.add(tf.keras.layers.Dropout(dr))
    m.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy'); return m

def run_cv(X, y, seed=SEED):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    fold_acc, oof = [], np.zeros(len(y))
    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        sc = MinMaxScaler().fit(X[tr]); Xtr, Xva = sc.transform(X[tr]), sc.transform(X[va])
        Xtr, ytr = SMOTE(random_state=seed).fit_resample(Xtr, y[tr])          # inside the fold only
        rf  = RandomForestClassifier(n_estimators=500, random_state=seed, n_jobs=1).fit(Xtr, ytr)
        xgb = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.01, subsample=0.9,
                            colsample_bytree=0.9, random_state=seed, n_jobs=1,
                            eval_metric='logloss', verbosity=0).fit(Xtr, ytr)
        dnn = build_dnn(Xtr.shape[1], seed + fold)
        dnn.fit(Xtr, ytr, epochs=80, batch_size=16, verbose=0, validation_split=0.15,
                callbacks=[tf.keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True)])
        p = (rf.predict_proba(Xva)[:, 1] + xgb.predict_proba(Xva)[:, 1] + dnn.predict(Xva, verbose=0).ravel()) / 3
        oof[va] = p; fold_acc.append(accuracy_score(y[va], (p >= .5).astype(int)))
        print(f"  fold {fold}: accuracy {fold_acc[-1]*100:.2f}%")
    return np.array(fold_acc), oof



In [ ]:
acc_c, oof_c = run_cv(X_c, y_c)
skf_c = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_c = np.zeros(len(y_c), int)
for f, (_, va) in enumerate(skf_c.split(X_c, y_c), 1): fold_c[va] = f
print(f"Coimbra CV check: {acc_c.mean()*100:.2f} +/- {acc_c.std(ddof=1)*100:.2f}   [paper: 70.62 +/- 14.35]")
C = {a: conformal_report(y_c, oof_c, fold_c, a, "Coimbra (116, cancer = positive)") for a in (0.05, 0.10)}


## ★ RESULTS SUMMARY

In [ ]:
print("="*78 + "\nNOTEBOOK 07 — RESULTS SUMMARY (is cell ka output bhejein)\n" + "="*78)
for name, R in (("WBCD", W), ("Coimbra", C)):
    for a, r in R.items():
        print(f"{name:<8} a={a:.2f} | in-sample {r['ins']*100:.2f}% | OOF marginal {r['cov']*100:.2f}% "
              f"(pos {r['cpos']*100:.2f}, neg {r['cneg']*100:.2f}) | sets {r['sets']} | OOF Mondrian pos {r['mpos']*100:.2f} neg {r['mneg']*100:.2f}")
